# LoRA Fine-tuning on a Single GPU

Educational notebook: adapt a pretrained causal language model using LoRA. Recommended starting model: **Qwen/Qwen2.5-0.5B-Instruct** (0.49B parameters, Apache-2.0 model card). Dataset: **databricks/databricks-dolly-15k**. This is a small supervised instruction-tuning experiment, not pretraining. Verified on MetaX C550 (64 GiB); the code uses only standard `torch.cuda` APIs and eager attention, so it is not tied to MetaX: an NVIDIA CUDA environment takes the same code path.

**PyTorch note:** do not install/upgrade PyTorch in this notebook. Use the vendor-supported build that ships with the image. We start in FP32 and use eager attention to reduce assumptions about specialized CUDA kernels. Mixed precision is optional and must be tested on your exact accelerator stack. Avoid bitsandbytes/QLoRA and vendor-specific fused optimizers until verified compatible.

## Prerequisites in a CubeStack DevEnvironment

Three sections below reach outside the cluster, and a DevEnvironment is restricted by
default: it resolves names but connects to nothing outside it. Enable outbound access
when you create the environment — **Advanced → Outbound access → Allow access outside
the cluster**, `spec.network.egress.unrestricted: true`:

- **§1** runs `%pip install`. `%pip` reports its own progress, not its failure: a
  blocked install surfaces much later, as a `ModuleNotFoundError` from whichever
  cell first imports what it installed.
- **§4** downloads `Qwen/Qwen2.5-0.5B-Instruct` from huggingface.co.
- **§5** downloads `databricks/databricks-dolly-15k` from huggingface.co.

Both downloads are cached under `~/.cache/huggingface`, so only the first run needs
the network.

If huggingface.co is unreachable or slow, point the downloads at a mirror by setting
`HF_ENDPOINT=https://hf-mirror.com` as an environment variable when you create the
environment (**Advanced → Environment variables**, `spec.runtime.env`). The mirror
serves the same Hub API, so `from_pretrained` and `load_dataset` use it unchanged.
`huggingface_hub` reads the variable when it is imported, so it must be in the
process environment before §4 — creating the environment with it guarantees that.

A mirror is a third party. If you have logged in to the Hub (or set `HF_TOKEN`) in this
environment, set `HF_HUB_DISABLE_IMPLICIT_TOKEN=1` beside it: without it
`huggingface_hub` sends your token to whatever endpoint is configured, mirror included.
With it, ordinary reads carry no token; a gated or private repository then needs an
explicit `token=` argument (or unset the variable and use huggingface.co directly).
Only use a mirror you trust with the requests themselves.

The remaining sections are compute-only.

## 1. Install Python packages
Run only if needed. Keep the vendor's PyTorch build intact.

In [ ]:
# Pins verified against the vendor image (Python 3.10, torch 2.4.0+metax3.9.0.8):
# transformers 5.x requires torch>=2.5 and disables PyTorch; datasets 5.x needs
# huggingface-hub>=1.31, which conflicts with transformers 4.x (<1.0).
%pip install transformers==4.57.6 datasets==4.3.0 peft==0.21.2 accelerate==1.15.0 safetensors==0.8.0 matplotlib==3.10.9 tqdm==4.67.1

print("✓ Cell 1 complete — dependencies installed.")

## 2. Verify the runtime

In [ ]:
import os, time, math, random, platform
import numpy as np
import torch
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("Device available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-compatible PyTorch build (NVIDIA CUDA or MetaX MACA) must expose the GPU through torch.cuda.")
print("Device:", torch.cuda.get_device_name(0))
device = torch.device("cuda:0")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print(f"✓ Cell 2 complete — GPU runtime verified on {torch.cuda.get_device_name(0)}.")

## 3. Configuration

0.5B is deliberately conservative for the first run. With 64 GiB, a 1.5B model is a reasonable later experiment, but first validate the full pipeline and operators. The defaults below are a short run (about half an hour on one C550); reduce `MAX_STEPS` for a smoke test.

Each execution writes into its own directory, `~/runs/<timestamp>/`, so a new run never overwrites an earlier one. The adapters and `run.json` live there; the base model and dataset stay in the shared Hugging Face cache.

In [ ]:
import json

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_ID = "databricks/databricks-dolly-15k"
MAX_LENGTH = 512
MAX_EXAMPLES = 5000
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
MAX_STEPS = 300
EVAL_INTERVAL = 50
EVAL_BATCHES = 10
LEARNING_RATE = 2e-4
WARMUP_STEPS = 20
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
USE_AMP = False                 # Enable only after the precision probe succeeds.
AMP_DTYPE = torch.bfloat16      # If probing FP16 instead, change this to torch.float16.

# Each execution gets its own directory under ~/runs, so a run never overwrites
# an earlier one. The directory is created exclusively: if two kernels start in
# the same second, the second takes a numeric suffix instead of sharing the
# first. The adapters and run.json are derived from RUN_DIR; the Hugging Face
# download cache stays shared (~/.cache/huggingface) so a re-run does not
# download the base model or dataset again.
#
# To reload an earlier run, set RESUME_RUN_ID to its directory name (for
# example "20261019-071600") and re-run this cell. Every path below then
# resolves inside that run, and its run.json is left untouched. Leave it None
# for a fresh run.
RESUME_RUN_ID = None
RUNS_DIR = os.path.join(os.path.expanduser("~"), "runs")
if RESUME_RUN_ID is not None:
    # A run name is a single path component: reject separators and ".." so a
    # mistyped value cannot point the checkpoint saves outside ~/runs.
    if RESUME_RUN_ID != os.path.basename(RESUME_RUN_ID) or RESUME_RUN_ID in ("", ".", ".."):
        raise ValueError(
            f"RESUME_RUN_ID must be a run directory name, got {RESUME_RUN_ID!r}"
        )
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = os.path.join(RUNS_DIR, RUN_ID)
    CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
    # Without this check a mistyped name quietly creates an empty run and the
    # failure only surfaces later, when the adapter cannot be loaded.
    if not os.path.isdir(CHECKPOINT_DIR):
        raise FileNotFoundError(f"No earlier run to reopen at {RUN_DIR}")
else:
    os.makedirs(RUNS_DIR, exist_ok=True)
    base = os.path.join(RUNS_DIR, time.strftime("%Y%m%d-%H%M%S"))
    RUN_DIR, suffix = base, 0
    while True:
        try:
            os.makedirs(RUN_DIR)   # exclusive: fails if the name is taken
            break
        except FileExistsError:
            suffix += 1
            RUN_DIR = f"{base}-{suffix}"
    RUN_ID = os.path.basename(RUN_DIR)
    CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
    os.makedirs(CHECKPOINT_DIR)

import transformers, peft
run_record = {
    "run_id": RUN_ID,
    "run_dir": RUN_DIR,
    "seed": SEED,
    "model": {
        "base_model": BASE_MODEL,
        "dtype": "float32",
        "attn_implementation": "eager",
    },
    "training": {
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "max_steps": MAX_STEPS,
        "eval_interval": EVAL_INTERVAL,
        "eval_batches": EVAL_BATCHES,
        "learning_rate": LEARNING_RATE,
        "warmup_steps": WARMUP_STEPS,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "use_amp": USE_AMP,
    },
    "data": {
        "dataset": DATASET_ID,
        "max_length": MAX_LENGTH,
        "max_examples": MAX_EXAMPLES,
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "device": torch.cuda.get_device_name(0),
    },
}
# A resumed run keeps the run.json written by the execution that produced it:
# rewriting it with today's values would describe a configuration that never
# produced the saved adapter.
if RESUME_RUN_ID is None:
    with open(os.path.join(RUN_DIR, "run.json"), "w", encoding="utf-8") as f:
        json.dump(run_record, f, indent=2)

print("Run directory:", RUN_DIR)
print(f"✓ Cell 3 complete — configuration set for run {RUN_ID} ({MAX_STEPS} steps).")

## 4. Load the tokenizer and base model
We explicitly use eager attention as a conservative compatibility starting point. If your Transformers version rejects this argument, remove it and test the default path.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
    attn_implementation="eager",
    trust_remote_code=False,
).to(device)
print("Vocab:", len(tokenizer))
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("GPU allocated GiB:", round(torch.cuda.memory_allocated()/1024**3, 2))

print("✓ Cell 4 complete — base model loaded on the GPU.")

## 5. Load the public dataset

Dolly-15k provides instruction/context/response records. Review the dataset card and terms before redistributing data or using the result commercially. We cap the number of examples to keep the first experiment manageable.

In [ ]:
from datasets import load_dataset
raw = load_dataset(DATASET_ID, split="train")
print(raw)
print(raw[0])

print(f"✓ Cell 5 complete — dataset loaded ({len(raw):,} examples).")

## 6. Format examples and mask prompt tokens

The model's chat template creates the prompt. Labels for the system/user prompt are masked with `-100`, so the loss trains on the assistant response rather than asking the model to reproduce the input prompt. Long examples are filtered out for simplicity.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."

def format_example(ex):
    user_text = ex.get("instruction", "") or ""
    context = ex.get("context", "")
    if context and str(context).strip():
        user_text += "\n\nContext:\n" + str(context).strip()
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    full_messages = prompt_messages + [
        {"role": "assistant", "content": ex.get("response", "") or ""}
    ]
    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True
    )
    full_ids = tokenizer.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False
    )
    return {"input_ids": full_ids, "prompt_len": len(prompt_ids)}

raw = raw.shuffle(seed=SEED).select(range(min(MAX_EXAMPLES, len(raw))))
formatted = raw.map(format_example, remove_columns=raw.column_names)
formatted = formatted.filter(
    lambda ex: len(ex["input_ids"]) <= MAX_LENGTH and len(ex["input_ids"]) > ex["prompt_len"]
)
splits = formatted.train_test_split(test_size=0.05, seed=SEED)
train_ds, val_ds = splits["train"], splits["test"]
print("Train:", len(train_ds), "Validation:", len(val_ds))
print(tokenizer.decode(train_ds[0]["input_ids"]))

print(f"✓ Cell 6 complete — data prepared ({len(train_ds):,} train / {len(val_ds):,} validation).")

## 7. Attach LoRA
The base model stays frozen. We train low-rank matrices attached to attention and MLP projections.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
for p in model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable_params = [p for p in model.parameters() if p.requires_grad]

print(f"✓ Cell 7 complete — LoRA attached ({sum(p.numel() for p in trainable_params):,} trainable parameters).")

## 8. Optional precision probe

This checks that autocast works for a real forward/backward pass. It does not guarantee that long training will be stable. Keep `USE_AMP=False` unless the probe succeeds and you have validated your accelerator stack. For a first educational run, FP32 is the safer baseline.

In [ ]:
def precision_probe(dtype):
    try:
        ex = train_ds[0]
        ids = torch.tensor([ex["input_ids"][:min(128, len(ex["input_ids"]))]],
                           dtype=torch.long, device=device)
        model.train()
        with torch.autocast(device_type="cuda", dtype=dtype, enabled=True):
            loss = model(input_ids=ids, labels=ids).loss
        loss.backward()
        model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        return True
    except Exception as e:
        model.zero_grad(set_to_none=True)
        print(dtype, "probe failed:", repr(e)[:500])
        return False

print("BF16 probe:", precision_probe(torch.bfloat16))
print("FP16 probe:", precision_probe(torch.float16))
print("USE_AMP currently:", USE_AMP)

print("✓ Cell 8 complete — precision probe finished.")

## 9. Batching and data loaders

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(examples):
    max_len = min(MAX_LENGTH, max(len(ex["input_ids"]) for ex in examples))
    ids_batch, labels_batch = [], []
    for ex in examples:
        ids = ex["input_ids"][:max_len]
        labels = list(ids)
        labels[:min(ex["prompt_len"], len(labels))] = [-100] * min(ex["prompt_len"], len(labels))
        pad = max_len - len(ids)
        ids_batch.append(ids + [tokenizer.pad_token_id] * pad)
        labels_batch.append(labels + [-100] * pad)
    ids = torch.tensor(ids_batch, dtype=torch.long)
    labels = torch.tensor(labels_batch, dtype=torch.long)
    mask = (ids != tokenizer.pad_token_id).long()
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device), "labels": labels.to(device)}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
batch = next(iter(train_loader))
print({k: tuple(v.shape) for k,v in batch.items()})
print("Response target tokens:", int((batch["labels"] != -100).sum()))

print("✓ Cell 9 complete — data loaders ready.")

## 10. Optimizer, evaluation, and adapter checkpointing

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.0)

@torch.no_grad()
def evaluate(loader, max_batches=EVAL_BATCHES):
    model.eval()
    losses = []
    for i, batch in enumerate(loader):
        if i >= max_batches: break
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = model(**batch).loss
        if torch.isfinite(loss):
            losses.append(float(loss.float().cpu()))
    model.train()
    return float(np.mean(losses)) if losses else float("nan")

def save_adapter(step, val_loss, name="latest"):
    path = os.path.join(CHECKPOINT_DIR, name)
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    torch.save({"step": step, "val_loss": val_loss,
                "optimizer_state_dict": optimizer.state_dict()},
               os.path.join(path, "training_state.pt"))
    return path

def lr_for_step(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)
    progress = min(1.0, (step-WARMUP_STEPS)/max(1, MAX_STEPS-WARMUP_STEPS))
    return LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi*progress)))

print("✓ Cell 10 complete — training helpers defined.")

## 11. Train

`MAX_STEPS=300` counts optimizer updates, each of which consumes `GRAD_ACCUM_STEPS` microbatches; it is a starter run, not a quality guarantee. Gradient accumulation increases the effective batch size without increasing per-microbatch memory. This minimal loop intentionally prioritizes transparency over maximum throughput.

In [ ]:
history = {"step": [], "train_loss": [], "val_loss": []}
model.train()
optimizer.zero_grad(set_to_none=True)
train_iter = iter(train_loader)
recent_losses = []
start_time = time.time()
pbar = tqdm(total=MAX_STEPS, desc="LoRA training")

# MAX_STEPS counts optimizer updates. Each update consumes GRAD_ACCUM_STEPS
# microbatches before clipping and stepping, so the learning-rate schedule,
# the evaluations and the checkpoints all line up with real updates, and no
# partially accumulated gradients are left unapplied at the end.
for step in range(1, MAX_STEPS + 1):
    for pg in optimizer.param_groups:
        pg["lr"] = lr_for_step(step - 1)

    step_loss = 0.0
    for _ in range(GRAD_ACCUM_STEPS):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = model(**batch).loss / GRAD_ACCUM_STEPS
        loss.backward()
        step_loss += float(loss.detach().float().cpu())
    recent_losses.append(step_loss)

    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    pbar.update(1)
    if step % 10 == 0:
        pbar.set_postfix(loss=f"{np.mean(recent_losses[-10:]):.3f}",
                         lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS:
        train_loss = float(np.mean(recent_losses[-EVAL_INTERVAL:]))
        val_loss = evaluate(val_loader)
        history["step"].append(step)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"step={step} train={train_loss:.4f} val={val_loss:.4f} elapsed_min={(time.time()-start_time)/60:.1f}")
        save_adapter(step, val_loss, "latest")
        if step % 100 == 0 or step == MAX_STEPS:
            save_adapter(step, val_loss, f"step-{step}")

pbar.close()
print("Elapsed minutes:", round((time.time()-start_time)/60, 1))

print(f"✓ Cell 11 complete — trained for {MAX_STEPS} optimizer steps.")

## 12. Plot the losses

In [ ]:
plt.figure(figsize=(8,4.5))
plt.plot(history["step"], history["train_loss"], marker="o", label="Train")
plt.plot(history["step"], history["val_loss"], marker="o", label="Validation")
plt.xlabel("Step"); plt.ylabel("Assistant-response cross-entropy")
plt.title("Qwen2.5-0.5B LoRA fine-tuning")
plt.grid(True, alpha=0.3); plt.legend(); plt.show()

print("✓ Cell 12 complete — loss plot rendered.")

## 13. Generate with the adapter

In [ ]:
model.eval()
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Explain LoRA in three simple bullet points."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=160, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.05,
                         pad_token_id=tokenizer.pad_token_id,
                         eos_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

print("✓ Cell 13 complete — generation finished.")

## 14. Export a merged, deployable model

An adapter is meant to be attached at load time: engines such as vLLM take a base model plus one or
more adapters (`--enable-lora`, `--lora-modules`) and swap them without reloading the base weights, so
each fine-tune costs a few megabytes and several adapters share one copy of the base. This notebook
takes the other route — it folds the delta into the base and saves a self-contained model — because
the export then loads anywhere with no adapter configuration, at the cost of one full copy per
fine-tune.

The adapter directory on its own is not loadable as a model. It carries only the low-rank delta and
the name of the base model it was trained against, so serving it requires a loader that knows to
attach both. Merging removes that requirement: the result is an ordinary Hugging Face directory, the
same shape as the tiny-gpt export.

`merge_and_unload()` folds `B @ A` into the target projections and returns a plain model with no peft
wrapper; the cell saves it to `~/runs/<id>/serving/`. Section 15 loads that directory back with the
stock `AutoModelForCausalLM` — no peft, no adapter reference — and generates once. That reload is what
proves the directory stands on its own.

Merging needs the whole model in memory: about 1 GB of fp32 weights for 0.5B, on top of the adapter.

In [ ]:
# Adapters are designed to load dynamically: an engine can hold the base model
# once and attach several adapters on top (vLLM: --enable-lora, --lora-modules).
# This notebook merges instead, so the export is a plain model directory that
# loads anywhere with no adapter config and no base model to match up.
# Fold the delta into the base and save a normal Hugging Face directory.
import os

SERVING_DIR = os.path.join(RUN_DIR, "serving")
os.makedirs(SERVING_DIR, exist_ok=True)
merged = model.merge_and_unload()
merged.eval()
merged.save_pretrained(SERVING_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(SERVING_DIR)

print("Exported to", SERVING_DIR)
for name in sorted(os.listdir(SERVING_DIR)):
    print("  %-30s %9.1f MiB" % (name, os.path.getsize(os.path.join(SERVING_DIR, name)) / 1024**2))
print(f"✓ Cell 14 complete — merged model exported to {SERVING_DIR}.")

## 15. Verify the export

`merge_and_unload()` returns the base model, and the PeftModel still holds the same
weights, so this cell drops both before loading the export back; otherwise the GPU
holds a second copy. It then reads `~/runs/<id>/serving/` with the stock
`AutoModelForCausalLM` — no peft, no adapter reference — and generates once, which is
what proves the directory stands on its own.

This releases the trained model, so section 13 cannot run again afterwards. To get the
adapter back, re-run section 4 and load the saved checkpoint:

```python
from peft import PeftModel
model = PeftModel.from_pretrained(model, os.path.join(CHECKPOINT_DIR, "latest")).to(device)
model.eval()
```

In [ ]:
# Read the export back with the stock class to prove it is self-contained: no
# peft, no adapter, just base weights that happen to contain the fine-tune.
#
# `model` is the PeftModel and `merged` is the base model it merged into, so
# both names still refer to the same GPU weights: dropping only `merged` keeps
# the trained model alive and the reload below would put a second copy on the
# GPU. Drop both, then let the allocator release them.
import gc

model = merged = None
gc.collect()
torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer

reloaded = AutoModelForCausalLM.from_pretrained(SERVING_DIR, torch_dtype=torch.float32).to(device)
reloaded.eval()
reload_tokenizer = AutoTokenizer.from_pretrained(SERVING_DIR)
prompt = reload_tokenizer.apply_chat_template(
    [{"role": "system", "content": SYSTEM_PROMPT},
     {"role": "user", "content": "Explain LoRA in three simple bullet points."}],
    tokenize=False, add_generation_prompt=True)
inputs = reload_tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = reloaded.generate(**inputs, max_new_tokens=80, do_sample=False,
                            pad_token_id=reload_tokenizer.pad_token_id,
                            eos_token_id=reload_tokenizer.eos_token_id)
print(reload_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

print(f"✓ Cell 15 complete — the export reloads on its own from {SERVING_DIR}.")

## 16. Reload later

In a fresh kernel, load the same base model and tokenizer first, then load the saved adapter. The adapter directory does not contain the base model.

To reopen an earlier run, set `RESUME_RUN_ID` in section 3 to its directory name under `~/runs/` and re-run sections 2, 3, and 4: section 2 defines the imports, `device` and `SEED` that section 3 uses, and section 4 loads the base model. Every path then resolves inside that run, so the adapter below is found without editing any path.

```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float32, attn_implementation="eager"
).to(device)
tuned = PeftModel.from_pretrained(base, os.path.join(CHECKPOINT_DIR, "latest")).to(device)
tuned.eval()
```

## Experiments and limitations

- Compare generations before and after training using the same prompts.
- Try LoRA rank 4, 8, 16; compare attention-only targets (`q_proj`, `v_proj`) with attention + MLP targets.
- After the 0.5B run works, try `Qwen/Qwen2.5-1.5B-Instruct`.
- Loss masking means the objective is on assistant responses only.
- A small dataset and a few hundred steps teach the mechanics, but won't make a general-purpose assistant.
- If an operator fails, use the full traceback and verify compatibility with your exact accelerator/PyTorch/Transformers versions. If autocast fails, leave FP32 enabled. Avoid bitsandbytes, FlashAttention-specific wheels, and CUDA-specific fused optimizers until confirmed supported.